# Step 7 (re-run) — Evaluasi komparatif LDA vs BERTopic dan topic alignment

**Input (semua dari step5 v2 dan step6 v2):**
`lda_grid_baseline_v2.csv`, `lda_baseline_vs_alt_v2.csv`, `df_lda_topics_v2.csv`, `lda_topics_final_v2.json`, `lda_representative_docs_v2.csv`,
`bertopic_grid_v2.csv`, `df_bertopic_topics_v2.csv`, `bertopic_topic_info_reduced_v2.csv`, `bertopic_representative_docs_v2.csv`, `embeddings_paraphrase-multilingual-MiniLM-L12-v2.npy`.

**Prinsip:**
1. Semua angka dibaca dari file hasil run, tidak ada yang diketik manual.
2. Perbandingan utama: **LDA K=25 vs BERTopic reduksi 25** (jumlah topik setara). BERTopic alami (121 topik) dilaporkan sebagai pembanding, bukan untuk diadu langsung, karena diversity dan coherence bergantung pada jumlah topik.
3. Kata "lebih tinggi" hanya dipakai jika rentang mean ± SD antar-seed tidak tumpang tindih. Dengan 3 seed, ini bukan uji signifikansi, hanya aturan kehati-hatian.
4. **Cakupan tidak sama:** LDA menugaskan 100% dokumen, BERTopic hanya dokumen non-outlier. Ini selalu dilaporkan berdampingan dengan coherence.
5. Interpretability tidak diklaim sebagai fakta empiris kecuali ada penilaian manusia (bagian E menyiapkan lembar penilaian buta).

In [ ]:
import os, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score, cohen_kappa_score
warnings.filterwarnings('ignore')

# ============ KONFIGURASI ============
K_LDA, K_BT = 25, 25
BASELINE_BT = 'mcs50'
STRONG_DOC_JACCARD  = 0.15   # ambang konvensi (bukan standar baku); dilaporkan apa adanya
STRONG_WORD_JACCARD = 0.20
RNG_SEED = 42
EMB_FILE = 'embeddings_paraphrase-multilingual-MiniLM-L12-v2.npy'
F = dict(lda_grid='lda_grid_baseline_v2.csv', lda_alt='lda_baseline_vs_alt_v2.csv', lda_docs='df_lda_topics_v2.csv',
         lda_topics='lda_topics_final_v2.json', lda_reps='lda_representative_docs_v2.csv',
         bt_grid='bertopic_grid_v2.csv', bt_docs='df_bertopic_topics_v2.csv',
         bt_info_red='bertopic_topic_info_reduced_v2.csv', bt_reps='bertopic_representative_docs_v2.csv', emb=EMB_FILE)
# ======================================
missing = [v for v in F.values() if not os.path.exists(v)]
print('File belum ada:', missing); assert not missing, 'Unggah file yang belum ada ke folder kerja.'
json.dump(dict(K_LDA=K_LDA, K_BT=K_BT, baseline_bt=BASELINE_BT, strong_doc_jaccard=STRONG_DOC_JACCARD,
               strong_word_jaccard=STRONG_WORD_JACCARD, files=F), open('comparison_config_v2.json', 'w'), indent=2)

## A. Muat data dan uji konsistensi (korpus LDA dan BERTopic harus identik baris-per-baris)

In [ ]:
lda_grid = pd.read_csv(F['lda_grid']); lda_alt = pd.read_csv(F['lda_alt']); bt_grid = pd.read_csv(F['bt_grid'])
dl = pd.read_csv(F['lda_docs']); db = pd.read_csv(F['bt_docs'])
assert len(dl) == len(db), f'Jumlah baris berbeda: LDA={len(dl)}, BERTopic={len(db)}'
assert (dl['title_raw'].astype(str).values == db['title_raw'].astype(str).values).all(), 'Urutan/isi title_raw berbeda antara file LDA dan BERTopic'
assert (dl['tahun'].values == db['tahun'].values).all()
d = pd.DataFrame({'tahun': dl['tahun'], 'jurnal': dl['jurnal'], 'title_raw': dl['title_raw'],
                  'lda': dl['dominant_topic'], 'lda_prob': dl['topic_prob'], 'lda_name': dl['lda_topic_name'],
                  'bt_nat': db['bertopic_id'], 'bt_red': db['bertopic_id_reduced'],
                  'bt_name': db['bertopic_name'] if 'bertopic_name' in db else db['bertopic_id_reduced'].astype(str)})
print(f"Dokumen = {len(d)} | tahun {d.tahun.min()}-{d.tahun.max()} | topik LDA = {d.lda.nunique()} | topik BERTopic (reduced, non-outlier) = {d.loc[d.bt_red!=-1,'bt_red'].nunique()}")
print('Korpus LDA dan BERTopic identik: OK')

## B. Metrik utama dan tabel komparatif (Tahap 12-13)

In [ ]:
def mean_sd(s):
    s = pd.Series(s).astype(float); return float(s.mean()), float(s.std(ddof=1)) if len(s) > 1 else 0.0

lda_k = lda_grid[lda_grid.K == K_LDA]
bt_b = bt_grid[bt_grid.config == BASELINE_BT]
groups = {f'LDA (K={K_LDA})': lda_k,
          f'BERTopic reduced ({K_BT})': bt_b[bt_b.variant == f'reduced_{K_BT}'],
          'BERTopic natural': bt_b[bt_b.variant == 'natural']}
rows = []
for name, g in groups.items():
    cm, cs = mean_sd(g.coherence_cv); dm, ds = mean_sd(g.diversity)
    r = dict(method=name, n_seeds=len(g), coh_mean=cm, coh_sd=cs, div_mean=dm, div_sd=ds)
    if name.startswith('LDA'):
        r.update(n_topics=float(K_LDA), outlier_pct_mean=0.0, outlier_pct_sd=0.0, perplexity_mean=float(g.perplexity.mean()))
    else:
        om, osd = mean_sd(g.outlier_pct); r.update(n_topics=float(g.n_topics.mean()), outlier_pct_mean=om, outlier_pct_sd=osd, perplexity_mean=np.nan)
    rows.append(r)
M = pd.DataFrame(rows); M.to_csv('comparative_metrics_v2.csv', index=False)
print(M.round(4).to_string(index=False))

def compare(a, am, asd, b, bm, bsd):
    if am - asd > bm + bsd: return f'{a} lebih tinggi (rentang mean±SD tidak tumpang tindih; n={len(lda_k)} seed)'
    if bm - bsd > am + asd: return f'{b} lebih tinggi (rentang mean±SD tidak tumpang tindih; n={len(lda_k)} seed)'
    return 'Tidak ada perbedaan yang jelas (rentang mean±SD tumpang tindih)'

L, R, N = M.iloc[0], M.iloc[1], M.iloc[2]
fmt = lambda m, s, p=4: f'{m:.{p}f} ± {s:.{p}f}'
cov_bt = 100 - R.outlier_pct_mean
table = pd.DataFrame([
    ['Number of topics', f'{K_LDA}', f'{int(R.n_topics)}', f'{N.n_topics:.0f} (rata-rata seed)', 'Reduced disamakan dengan K LDA; natural ditentukan data'],
    ['Coherence (C_v)', fmt(L.coh_mean, L.coh_sd), fmt(R.coh_mean, R.coh_sd), fmt(N.coh_mean, N.coh_sd),
     'LDA vs BT-reduced: ' + compare('LDA', L.coh_mean, L.coh_sd, 'BERTopic', R.coh_mean, R.coh_sd)],
    ['Topic diversity', fmt(L.div_mean, L.div_sd), fmt(R.div_mean, R.div_sd), fmt(N.div_mean, N.div_sd),
     'LDA vs BT-reduced: ' + compare('LDA', L.div_mean, L.div_sd, 'BERTopic', R.div_mean, R.div_sd) + '. Diversity BT-natural tidak sebanding langsung (jumlah topik berbeda)'],
    ['Outlier rate', 'N/A (semua dokumen ditugaskan)', fmt(R.outlier_pct_mean, R.outlier_pct_sd, 2) + ' %', fmt(N.outlier_pct_mean, N.outlier_pct_sd, 2) + ' %',
     f'Cakupan: LDA 100% vs BERTopic {cov_bt:.1f}% dokumen. Coherence BERTopic dihitung pada topik dari dokumen non-outlier'],
    ['Perplexity', f'{L.perplexity_mean:.1f}', 'N/A', 'N/A', 'Hanya relevan untuk LDA; bukan kriteria antar-metode'],
    ['Interpretability', 'Penilaian kualitatif (lihat bagian E)', 'Penilaian kualitatif (lihat bagian E)', '-', 'Bukan fakta empiris kecuali penilaian manusia dilakukan'],
    ['Computational consideration', 'CPU; tanpa embedding', 'Embedding + UMAP + HDBSCAN (GPU disarankan)', 'sama', 'Kualitatif. Waktu tidak dibandingkan: konfigurasi proses berbeda (COH_PROCESSES) antara notebook'],
], columns=['Criterion', f'LDA (K={K_LDA})', f'BERTopic reduced ({K_BT})', 'BERTopic natural', 'Observation'])
table.to_csv('comparative_table_v2.csv', index=False)
pd.set_option('display.max_colwidth', 120); print(table.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0))
labels = [f'LDA\n(K={K_LDA})', f'BERTopic\nreduced ({K_BT})', 'BERTopic\nnatural']
cols = ['#4C72B0', '#DD8452', '#DD8452']; hatches = ['', '', '//']
for ax, (m, s, yl) in zip(axes, [('coh_mean', 'coh_sd', 'Coherence (C_v)'), ('div_mean', 'div_sd', 'Topic diversity')]):
    for k in range(3):
        ax.bar(k, M.iloc[k][m], yerr=M.iloc[k][s], color=cols[k], hatch=hatches[k], capsize=4, edgecolor='black', lw=.6)
    ax.set_xticks(range(3)); ax.set_xticklabels(labels); ax.set_ylabel(yl); ax.grid(alpha=.3, axis='y')
axes[0].set_title('Coherence'); axes[1].set_title('Diversity')
fig.suptitle(f'Figure 9. LDA vs BERTopic (mean ± SD over {len(lda_k)} seeds). BERTopic covers {cov_bt:.1f}% of titles; LDA covers 100%', fontsize=10)
fig.tight_layout(); plt.savefig('figure9_method_comparison_v2.png', dpi=300, bbox_inches='tight'); plt.show()

## C. Distribusi ukuran topik (Tahap 12C)

In [ ]:
def size_stats(labels, name, total, exclude=None):
    s = pd.Series(labels).value_counts()
    if exclude is not None: s = s.drop(index=exclude, errors='ignore')
    x = np.sort(s.values.astype(float)); n = len(x); p = x / x.sum()
    gini = (2 * np.sum(np.arange(1, n + 1) * x) / (n * x.sum())) - (n + 1) / n
    ent = -(p * np.log(p)).sum() / np.log(n) if n > 1 else np.nan
    return dict(method=name, n_topics=n, docs_assigned=int(x.sum()), coverage_pct=100 * x.sum() / total,
                size_median=float(np.median(x)), size_min=int(x.min()), size_max=int(x.max()),
                largest_share_pct_of_assigned=100 * x.max() / x.sum(), gini=gini, norm_entropy=ent)

N_all = len(d)
cov = (d.bt_red != -1).mean()
thr = np.quantile(d.lda_prob, 1 - cov)                       # LDA: ambil dokumen paling yakin agar cakupan sama dengan BERTopic
mask = d.lda_prob >= thr
S = pd.DataFrame([size_stats(d.lda, f'LDA (K={K_LDA})', N_all),
                  size_stats(d.lda[mask], f'LDA (K={K_LDA}), coverage-matched (prob >= {thr:.3f})', N_all),
                  size_stats(d.bt_red, f'BERTopic reduced ({K_BT})', N_all, exclude=-1),
                  size_stats(d.bt_nat, 'BERTopic natural', N_all, exclude=-1)])
S.to_csv('topic_size_stats_v2.csv', index=False); print(S.round(3).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for lab, sizes, st in [(f'LDA (K={K_LDA})', d.lda.value_counts(), '-o'),
                       (f'BERTopic reduced ({K_BT})', d.bt_red[d.bt_red != -1].value_counts(), '-s'),
                       ('BERTopic natural', d.bt_nat[d.bt_nat != -1].value_counts(), '--')]:
    v = np.sort(sizes.values)[::-1]; ax.plot(range(1, len(v) + 1), v, st, ms=3.5, lw=1.2, label=lab)
ax.set_yscale('log'); ax.set_xscale('log'); ax.set_xlabel('Topic rank by size (log)'); ax.set_ylabel('Number of titles (log)')
ax.grid(alpha=.3); ax.legend(frameon=False)
ax.set_title('Figure 10. Topic size distribution (rank-size plot)', fontsize=10)
fig.tight_layout(); plt.savefig('figure10_topic_size_distribution_v2.png', dpi=300, bbox_inches='tight'); plt.show()

## D. Topic alignment (Tahap 14)
Tiga ukuran, semuanya dihitung dari data (bukan dari nomor topik):
1. **Kemiripan semantik:** cosine antara centroid embedding judul dokumen tiap topik (ruang embedding yang sama).
2. **Overlap dokumen:** Jaccard himpunan dokumen (hanya dokumen yang ditugaskan BERTopic).
3. **Overlap kata:** Jaccard himpunan kata (top-10 frasa dipecah menjadi kata).

**Definisi pasangan kuat** (ditetapkan sebelum melihat hasil): *mutual best match* pada cosine, DAN (doc-Jaccard >= ambang ATAU word-Jaccard >= ambang). Topik tanpa pasangan kuat dilaporkan sebagai tidak memiliki counterpart yang jelas. Karena semua judul berbidang komputer, cosine antar-centroid cenderung tinggi secara umum; karena itu cosine dipakai untuk peringkat, bukan sebagai bukti tunggal.

In [ ]:
emb = np.load(F['emb']); assert emb.shape[0] == len(d), 'Jumlah baris embedding berbeda dari korpus'
lda_ids = sorted(d.lda.unique()); bt_ids = sorted(set(d.bt_red.unique()) - {-1})

def centroids(lbl, ids):
    Cm = np.vstack([emb[lbl == i].mean(axis=0) for i in ids]); return Cm / np.linalg.norm(Cm, axis=1, keepdims=True)
COS = pd.DataFrame(centroids(d.lda.values, lda_ids) @ centroids(d.bt_red.values, bt_ids).T, index=lda_ids, columns=bt_ids)

A = d[d.bt_red != -1]
CT = pd.crosstab(A.lda, A.bt_red).reindex(index=lda_ids, columns=bt_ids, fill_value=0)
nL, nB = CT.sum(axis=1).values[:, None], CT.sum(axis=0).values[None, :]
JAC = pd.DataFrame(CT.values / (nL + nB - CT.values), index=lda_ids, columns=bt_ids)
FRAC_L = CT.div(CT.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)   # porsi dokumen suatu topik LDA yang jatuh ke tiap topik BT
FRAC_B = CT.div(CT.sum(axis=0).replace(0, np.nan), axis=1).fillna(0)   # porsi dokumen suatu topik BT yang jatuh ke tiap topik LDA

lda_words = json.load(open(F['lda_topics']))
bt_info = pd.read_csv(F['bt_info_red']); bt_info = bt_info[bt_info.topic != -1]
uni = lambda phrases: set(w for p in phrases for w in str(p).split())
Lw = {int(k): uni(v) for k, v in lda_words.items()}
Bw = {int(r.topic): uni(str(r.top_words).split(', ')) for r in bt_info.itertuples()}
WJ = pd.DataFrame([[len(Lw[i] & Bw[j]) / len(Lw[i] | Bw[j]) for j in bt_ids] for i in lda_ids], index=lda_ids, columns=bt_ids)

nm_l = d.groupby('lda')['lda_name'].first(); nm_b = A.groupby('bt_red')['bt_name'].first()
best_b = COS.idxmax(axis=1); best_l = COS.idxmax(axis=0)
rows = []
for i in lda_ids:
    j = best_b[i]; mutual = bool(best_l[j] == i); dj, wj = JAC.loc[i, j], WJ.loc[i, j]
    rows.append(dict(lda_topic=i, lda_name=nm_l[i], bt_topic=j, bt_name=nm_b[j], cosine=COS.loc[i, j], doc_jaccard=dj, word_jaccard=wj,
                     share_of_lda_topic_in_bt=FRAC_L.loc[i, j], share_of_bt_topic_in_lda=FRAC_B.loc[i, j],
                     best_bt_by_docs=JAC.loc[i].idxmax(), mutual_best_cosine=mutual,
                     strong_match=bool(mutual and (dj >= STRONG_DOC_JACCARD or wj >= STRONG_WORD_JACCARD))))
AL = pd.DataFrame(rows); AL.to_csv('topic_alignment_v2.csv', index=False)
for nm, M_ in [('cosine', COS), ('doc_jaccard', JAC), ('word_jaccard', WJ), ('contingency', CT)]: M_.to_csv(f'alignment_{nm}_v2.csv')

pd.set_option('display.width', 250)
print(AL[['lda_topic', 'lda_name', 'bt_topic', 'bt_name', 'cosine', 'doc_jaccard', 'word_jaccard', 'mutual_best_cosine', 'strong_match']].round(3).to_string(index=False))
print(f"\nPasangan kuat: {AL.strong_match.sum()} dari {len(AL)} topik LDA")
strong_bt = set(AL[AL.strong_match].bt_topic)
print('Topik BERTopic tanpa pasangan kuat:', [(int(j), nm_b[j]) for j in bt_ids if j not in strong_bt])
print('Topik LDA tanpa pasangan kuat     :', [(r.lda_topic, r.lda_name) for r in AL[~AL.strong_match].itertuples()])

nmi = normalized_mutual_info_score(A.lda, A.bt_red); ari = adjusted_rand_score(A.lda, A.bt_red)
nmi_nat = normalized_mutual_info_score(A.lda, A.bt_nat)
agree = dict(n_docs_compared=int(len(A)), nmi_lda_vs_bt_reduced=nmi, ari_lda_vs_bt_reduced=ari, nmi_lda_vs_bt_natural=nmi_nat,
             n_strong=int(AL.strong_match.sum()), n_lda_topics=len(AL))
json.dump(agree, open('partition_agreement_v2.json', 'w'), indent=2); print(agree)

In [ ]:
am = FRAC_L.idxmax(axis=1)
order_l = sorted(lda_ids, key=lambda i: (am[i], -FRAC_L.loc[i].max()))
order_b = list(dict.fromkeys([am[i] for i in order_l])) + [j for j in bt_ids if j not in set(am)]
fig, ax = plt.subplots(figsize=(9, 7.2))
im = ax.imshow(FRAC_L.loc[order_l, order_b].values, cmap='Blues', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(order_b))); ax.set_xticklabels([f'B{j}' for j in order_b], rotation=90, fontsize=8)
ax.set_yticks(range(len(order_l))); ax.set_yticklabels([f'L{i}' for i in order_l], fontsize=8)
ax.set_xlabel('BERTopic topic (reduced)'); ax.set_ylabel('LDA topic')
cb = fig.colorbar(im, ax=ax); cb.set_label('Share of LDA topic titles (BERTopic-assigned only)')
ax.set_title('Figure 11. Document-level overlap between LDA and BERTopic topics', fontsize=10)
fig.tight_layout(); plt.savefig('figure11_topic_alignment_heatmap_v2.png', dpi=300, bbox_inches='tight'); plt.show()

## E. Interpretability: lembar penilaian buta (opsional, hanya jika ada penilai manusia)
Lembar berisi 50 topik (25 LDA + 25 BERTopic) yang diacak tanpa label metode. Minta 2-3 penilai memberi skor 1-5 (1 = sangat buruk, 5 = sangat baik) untuk: *apakah top words membentuk tema yang koheren dan mudah ditafsirkan*. Jika tidak ada penilai, laporkan interpretability sebagai **penilaian kualitatif**, bukan hasil empiris.

In [ ]:
lr = pd.read_csv(F['lda_reps']); br = pd.read_csv(F['bt_reps']); bt_iw = bt_info.set_index('topic')['top_words']
items = []
for i in lda_ids:
    t = lr[lr.dominant_topic == i].sort_values('topic_prob', ascending=False).title_raw.head(5).astype(str).tolist()
    items.append(dict(method='LDA', topic=i, name=nm_l[i], top_words='; '.join(lda_words[str(i)]), sample_titles=' | '.join(t)))
for j in bt_ids:
    t = br[br.topic == j].title_raw.head(5).astype(str).tolist()
    items.append(dict(method='BERTopic', topic=j, name=nm_b[j], top_words='; '.join(str(bt_iw[j]).split(', ')), sample_titles=' | '.join(t)))
it = pd.DataFrame(items).sample(frac=1, random_state=RNG_SEED).reset_index(drop=True)
it['item_id'] = [f'I{k+1:02d}' for k in range(len(it))]
it[['item_id', 'method', 'topic', 'name']].to_csv('interpretability_key_v2.csv', index=False)          # KUNCI: jangan diberikan ke penilai
blind = it[['item_id', 'top_words', 'sample_titles']].copy(); blind['score_1to5'] = ''; blind['comment'] = ''
blind.to_csv('interpretability_rubric_blind_v2.csv', index=False)
print('Lembar penilai:', len(blind), 'item -> interpretability_rubric_blind_v2.csv')

In [ ]:
# Jalankan setelah penilai mengisi. Format file: item_id, rater, score  (satu baris per penilaian)
SC = 'interpretability_scores_filled.csv'
if os.path.exists(SC):
    sc = pd.read_csv(SC).merge(pd.read_csv('interpretability_key_v2.csv'), on='item_id')
    print(sc.groupby('method')['score'].agg(['mean', 'std', 'count']).round(3))
    raters = sorted(sc.rater.unique())
    if len(raters) == 2:
        w = sc.pivot_table(index='item_id', columns='rater', values='score')
        print('Quadratic-weighted kappa:', round(cohen_kappa_score(w[raters[0]], w[raters[1]], weights='quadratic'), 3))
else:
    print('Belum ada skor penilai -> interpretability dilaporkan sebagai penilaian kualitatif.')

## F. Ringkasan robustness (Tahap 19): baseline, alternatif, selisih teramati. Kolom *conclusion* diisi manual setelah membaca

In [ ]:
rob = []
def add(area, base, alt, diff): rob.append(dict(area=area, baseline=base, alternative=alt, observed_difference=diff, conclusion=''))
s = lda_grid.groupby('K').agg(coh_m=('coherence_cv', 'mean'), div_m=('diversity', 'mean'))
for K2 in [20, 30]:
    if K2 in s.index: add('LDA: jumlah topik', f'K={K_LDA}', f'K={K2}', f'coherence {s.coh_m[K2]-s.coh_m[K_LDA]:+.4f}; diversity {s.div_m[K2]-s.div_m[K_LDA]:+.4f}')
add('LDA: seed (3 seed)', f'K={K_LDA}', 'seed 42/123/2024', f'SD coherence = {lda_k.coherence_cv.std():.4f}; rentang {lda_k.coherence_cv.min():.4f}-{lda_k.coherence_cv.max():.4f}')
a = lda_alt[lda_alt.K == K_LDA]
if len(a): add('LDA: preprocessing (stopword domain)', 'baseline', 'domain stopwords dibuang', f'coherence {a.iloc[0].delta_coh:+.4f}; diversity {a.iloc[0].delta_div:+.4f} (C_v dihitung pada korpus masing-masing)')
for v in ['natural', f'reduced_{K_BT}']:
    g = bt_b[bt_b.variant == v]
    add(f'BERTopic: seed ({v})', BASELINE_BT, 'seed 42/123/2024', f'topik {int(g.n_topics.min())}-{int(g.n_topics.max())}; outlier SD {g.outlier_pct.std():.2f} pt; coherence SD {g.coherence_cv.std():.4f}; diversity SD {g.diversity.std():.4f}')
s0 = bt_grid.seed.min()
base = bt_grid[(bt_grid.config == BASELINE_BT) & (bt_grid.seed == s0) & (bt_grid.variant == 'natural')].iloc[0]
for r in bt_grid[(bt_grid.config != BASELINE_BT) & (bt_grid.variant == 'natural')].itertuples():
    diff = (f'topik {int(r.n_topics)} (vs {int(base.n_topics)}); outlier {r.outlier_pct:.1f}% (vs {base.outlier_pct:.1f}%); '
            f'coherence {r.coherence_cv-base.coherence_cv:+.4f}; diversity {r.diversity-base.diversity:+.4f}')
    if r.n_topics < 5: diff += '  [DEGENERATE: <5 topik, satu topik mendominasi; tidak dipakai untuk kesimpulan]'
    add('BERTopic: konfigurasi klaster', f'{BASELINE_BT} (seed {s0})', r.config, diff)
R_ = pd.DataFrame(rob); R_.to_csv('robustness_summary_v2.csv', index=False)
pd.set_option('display.max_colwidth', 150); print(R_.drop(columns='conclusion').to_string(index=False))

## G. File yang dihasilkan
`comparison_config_v2.json`, `comparative_metrics_v2.csv`, `comparative_table_v2.csv`, `topic_size_stats_v2.csv`, `topic_alignment_v2.csv`,
`alignment_cosine_v2.csv`, `alignment_doc_jaccard_v2.csv`, `alignment_word_jaccard_v2.csv`, `alignment_contingency_v2.csv`, `partition_agreement_v2.json`,
`robustness_summary_v2.csv`, `interpretability_rubric_blind_v2.csv`, `interpretability_key_v2.csv`,
`figure9_method_comparison_v2.png`, `figure10_topic_size_distribution_v2.png`, `figure11_topic_alignment_heatmap_v2.png`.